# The Ultimate OpenCV 7-Day Bootcamp: Professional Mastery
Welcome to the comprehensive, professional-grade OpenCV training notebook. This notebook is designed to transition you from a beginner to a computer vision professional ready for the final round of your competition.

**Roadmap Overview:**
* **Day 1:** Image Processing Basics, Color Spaces, Gamma Correction & LUTs
* **Day 2:** Filtering, Spatial Convolutions, Custom Kernels & Non-Local Means Denoising
* **Day 3:** Edge Detection, Image Gradients (Magnitude/Angle) & Auto-Canny Pipeline
* **Day 4:** Morphology, Top-Hat Transforms, Contours & Convex Hulls
* **Day 5:** Hough Lines, Hough Circles & Robust Line Fitting (M-Estimators)
* **Day 6:** Perspective Transform, Feature Matching (ORB/SIFT) & Automated Homography via RANSAC
* **Day 7:** Phase 1 Project (End-to-End Expert Pipeline)


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import urllib.request
import math
import random

# Create directory for images
os.makedirs('data/images', exist_ok=True)


## Day 1 — Image Processing Basics & Expert Techniques
**Topics:** File I/O, Color Spaces, ROI, Histograms, Gamma Correction, Bitwise Masks.

### 1. Reusable Image I/O & Core Operations
*   **`cv2.imread(filepath, flag)`**: Loads an image tensor $(H, W, C)$. OpenCV reads color in **BGR** format.
*   **`cv2.cvtColor(src, code)`**: Translates between color spaces. **HSV** (Hue, Saturation, Value) is critical for object tracking because it decouples color (Hue) from lighting intensity (Value).
*   **`cv2.resize(src, dsize, fx, fy, interpolation)`**: 
    *   `cv2.INTER_AREA`: Mathematically resamples using pixel area relation. The industry standard for shrinking (downsampling) without aliasing.
    *   `cv2.INTER_LINEAR`: Bilinear interpolation. Standard for zooming.


In [ ]:
def download_image(url, filename):
    '''Downloads test images if they don't already exist.'''
    filepath = os.path.join('data/images', filename)
    if not os.path.exists(filepath):
        urllib.request.urlretrieve(url, filepath)
    return filepath

def load_image(filepath, as_rgb=True):
    '''Loads an image safely. Converts to RGB by default.'''
    img = cv2.imread(filepath)
    if img is None: raise ValueError(f"Not found: {filepath}")
    if as_rgb: img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def show_image(img, title="", cmap=None, figsize=(6, 6)):
    plt.figure(figsize=figsize); plt.imshow(img, cmap=cmap); plt.title(title); plt.axis('off'); plt.show()

def show_images_grid(images, titles, rows, cols, figsize=(15, 10), cmap=None):
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    for i, (img, title) in enumerate(zip(images, titles)):
        axes.flatten()[i].imshow(img, cmap=cmap); axes.flatten()[i].set_title(title); axes.flatten()[i].axis('off')
    plt.tight_layout(); plt.show()

def resize_keep_ratio(img, target_width=None, target_height=None):
    h, w = img.shape[:2]
    if target_width: ratio = target_width / w; dim = (target_width, int(h * ratio))
    else: ratio = target_height / h; dim = (int(w * ratio), target_height)
    return cv2.resize(img, dim, interpolation=cv2.INTER_AREA)

urls = [
    ("https://raw.githubusercontent.com/opencv/opencv/master/samples/data/messi5.jpg", "messi.jpg"),
    ("https://raw.githubusercontent.com/opencv/opencv/master/samples/data/sudoku.png", "sudoku.jpg"),
    ("https://raw.githubusercontent.com/opencv/opencv/master/samples/data/smarties.png", "smarties.jpg")
]
image_paths = [download_image(url, name) for url, name in urls]


### 2. Expert Addition: LUTs, Gamma Correction & Bitwise Masking
In professional CV, adjusting brightness/contrast linearly is insufficient because human vision and camera sensors are non-linear. 
*   **Gamma Correction via `cv2.LUT` (Look Up Table):** Instead of looping over millions of pixels (which is extremely slow in Python), professionals map values using an $O(1)$ lookup table matrix. $	ext{Output} = 255 	imes (rac{	ext{Input}}{255})^{1/\gamma}$.
*   **Masking (`cv2.inRange` & `cv2.bitwise_and`):** Isolating objects based on color logic.


In [ ]:
def adjust_gamma(image, gamma=1.0):
    '''Applies non-linear Gamma Correction using a fast Look-Up Table (LUT).'''
    invGamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
    return cv2.LUT(image, table) # O(1) pixel transformation

img_messi = load_image(image_paths[0])
img_dark = adjust_gamma(img_messi, gamma=0.4) # Darken
img_bright = adjust_gamma(img_messi, gamma=2.5) # Brighten

# Masking: Extracting the Smarties based on HSV
img_smarties = load_image(image_paths[2])
hsv_smarties = cv2.cvtColor(img_smarties, cv2.COLOR_RGB2HSV)

# Define blue color bounds in HSV
lower_blue = np.array([110, 50, 50])
upper_blue = np.array([130, 255, 255])

# Create binary mask and apply Bitwise AND
mask = cv2.inRange(hsv_smarties, lower_blue, upper_blue)
isolated_blue = cv2.bitwise_and(img_smarties, img_smarties, mask=mask)

show_images_grid(
    [img_messi, img_dark, img_bright, isolated_blue],
    ['Original', 'Gamma = 0.4 (Dark)', 'Gamma = 2.5 (Bright)', 'Bitwise Color Masking'],
    2, 2, figsize=(10, 8)
)


## Day 2 — Filtering & Thresholding (Expert Algorithms)
**Topics:** Standard Blurs, Non-Local Means, Custom Convolution Kernels.

### 1. Spatial Filtering (Convolution) & Custom Kernels
Filtering relies on **Convolution**—sliding a small matrix (a *kernel*) over the image to compute weighted local sums.
*   **`cv2.GaussianBlur`**: Blurs based on a 2D normal distribution. Perfect for Gaussian sensor noise.
*   **`cv2.medianBlur`**: Replaces the center pixel with the local median. The mathematical nemesis of Salt & Pepper noise.
*   **`cv2.bilateralFilter`**: Weights neighbors by both spatial distance AND color similarity. Preserves hard edges.
*   **`cv2.filter2D(src, ddepth, kernel)`**: Allows you to apply *any* arbitrary convolution matrix. We will build a custom Sharpening Kernel.


In [ ]:
def add_sp_noise(image, prob=0.05):
    noisy = np.copy(image); num = int(prob * image.size * 0.5)
    noisy[tuple([np.random.randint(0, i-1, num) for i in image.shape[:2]])] = 255
    noisy[tuple([np.random.randint(0, i-1, num) for i in image.shape[:2]])] = 0
    return noisy

img_gray = cv2.cvtColor(img_messi, cv2.COLOR_RGB2GRAY)
img_noisy = add_sp_noise(img_gray, prob=0.05)

# Standard filters
f_gauss = cv2.GaussianBlur(img_noisy, (5, 5), 0)
f_median = cv2.medianBlur(img_noisy, 5) # Eliminates S&P entirely

# Custom Sharpening Kernel via filter2D
# Sum of weights = 1 to maintain overall image brightness
sharpen_kernel = np.array([[ 0, -1,  0],
                           [-1,  5, -1],
                           [ 0, -1,  0]])
img_sharpened = cv2.filter2D(img_gray, -1, sharpen_kernel)

show_images_grid(
    [img_noisy, f_gauss, f_median, img_sharpened],
    ['S&P Noise', 'Gauss (Fails)', 'Median (Succeeds)', 'Custom Sharpen Kernel'],
    2, 2, cmap='gray'
)


### 2. Expert Addition: Non-Local Means Denoising
*   **`cv2.fastNlMeansDenoising(src, h, templateWindowSize, searchWindowSize)`**
*   **Concept:** While a Bilateral filter looks at the immediate neighborhood, NLM looks at the *entire image* (or a large search window) to find patches of pixels that look structurally similar to the current patch, and averages them together. 
*   **CV Use Case:** The industry gold-standard for removing noise from high-quality images where preserving texture (like skin or fabric) is absolutely critical.


In [ ]:
noisy_messi_color = add_sp_noise(img_messi, prob=0.02)
# NLM Denoising is computationally heavy but incredibly effective
nlm_denoised = cv2.fastNlMeansDenoisingColored(noisy_messi_color, None, 10, 10, 7, 21)

show_images_grid([noisy_messi_color, nlm_denoised], ['Noisy Input', 'Non-Local Means Output'], 1, 2)


## Day 3 — Edge Detection & Gradients
**Topics:** Sobel, Scharr, Gradient Magnitude/Angle, Auto-Canny.

### 1. Image Derivatives: Magnitude & Orientation
An edge is a rapid change in intensity. We calculate the derivative of the image using matrices like Sobel.
*   **`cv2.cartToPolar(x, y, angleInDegrees)`**: Once we have the Sobel $X$ and $Y$ gradients, we can compute the exact magnitude (strength) and angle (direction) of the edge.
*   **CV Use Case:** Extracting gradient orientations is the mathematical foundation of advanced object detection algorithms like HOG (Histogram of Oriented Gradients) used in pedestrian detection.


In [ ]:
img_sudoku_gray = cv2.cvtColor(load_image(image_paths[1]), cv2.COLOR_RGB2GRAY)

sobel_x = cv2.Sobel(img_sudoku_gray, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(img_sudoku_gray, cv2.CV_64F, 0, 1, ksize=3)

# Expert: Calculate strict mathematical magnitude and direction
magnitude, angle = cv2.cartToPolar(sobel_x, sobel_y, angleInDegrees=True)

show_images_grid(
    [sobel_x, sobel_y, magnitude, angle],
    ['Sobel X (Vertical Edges)', 'Sobel Y (Horizontal Edges)', 'Gradient Magnitude', 'Gradient Angle'],
    2, 2, cmap='gray'
)


### 2. Expert Addition: Auto-Canny Algorithm
The Canny pipeline requires manual lower/upper thresholds (`threshold1`, `threshold2`). Hardcoding thresholds destroys robustness across different lighting conditions.
*   **Auto-Canny Concept:** We compute the median pixel intensity $v$. We dynamically set the lower bounds to $(1 - \sigma) 	imes v$ and upper bounds to $(1 + \sigma) 	imes v$. This makes the algorithm robust to dynamic lighting.


In [ ]:
def auto_canny(image, sigma=0.33):
    '''Dynamically calculates Canny thresholds based on the image's median intensity.'''
    v = np.median(image)
    lower = int(max(0, (1.0 - sigma) * v))
    upper = int(min(255, (1.0 + sigma) * v))
    return cv2.Canny(image, lower, upper)

hardcoded_canny = cv2.Canny(img_sudoku_gray, 50, 150)
automated_canny = auto_canny(img_sudoku_gray)

show_images_grid([hardcoded_canny, automated_canny], ['Hardcoded Canny', 'Auto-Canny (Dynamic)'], 1, 2, cmap='gray')


## Day 4 — Morphology & Contours (Structural Analysis)
**Topics:** Erosion, Dilation, Top-Hat, Black-Hat, Contours, Rotated Rects, Convex Hulls.

### 1. Advanced Morphology: Top-Hat & Black-Hat
Morphology uses a Structuring Element (a logical kernel) to probe the image.
*   **Top-Hat (`cv2.MORPH_TOPHAT`)**: $Original - Opening$. Extracts bright elements smaller than the structuring element. **Use Case:** Fixing uneven illumination (e.g., reading text on a heavily shadowed page).
*   **Black-Hat (`cv2.MORPH_BLACKHAT`)**: $Closing - Original$. Extracts dark elements smaller than the structuring element.


In [ ]:
# Create synthetic gradient background with text
gradient_bg = np.tile(np.linspace(0, 150, 400, dtype=np.uint8), (400, 1)).T
cv2.putText(gradient_bg, "OpenCV PRO", (50, 200), cv2.FONT_HERSHEY_SIMPLEX, 1.5, 255, 3)

# Top Hat Transform to extract text despite gradient lighting
kernel_rect = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 25))
tophat = cv2.morphologyEx(gradient_bg, cv2.MORPH_TOPHAT, kernel_rect)

show_images_grid([gradient_bg, tophat], ['Uneven Illumination', 'Top-Hat Extract (Text isolated)'], 1, 2, cmap='gray')


### 2. Advanced Contours: Convex Hull & Rotated Rects
*   **`cv2.minAreaRect(contour)`**: Finds the minimum bounding rectangle enclosing the shape, even if the object is rotated diagonally. Extremely useful for measuring real-world objects.
*   **`cv2.convexHull(contour)`**: Finds the convex envelope (imagine stretching a rubber band around the shape). Used in gesture recognition (hand contours vs hull yields fingertips).


In [ ]:
# Create synthetic rotated object
img_shape = np.zeros((300, 300), dtype=np.uint8)
pts = np.array([[100, 50], [200, 100], [150, 250], [50, 200]], np.int32)
cv2.fillPoly(img_shape, [pts], 255)

contours, _ = cv2.findContours(img_shape, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cnt = contours[0]

# Standard vs Rotated Bounding Box
x, y, w, h = cv2.boundingRect(cnt) # Straight rect
rect = cv2.minAreaRect(cnt)        # Rotated rect
box = cv2.boxPoints(rect)
box = np.int64(box)

# Convex Hull
hull = cv2.convexHull(cnt)

output = cv2.cvtColor(img_shape, cv2.COLOR_GRAY2BGR)
cv2.rectangle(output, (x, y), (x+w, y+h), (255, 0, 0), 2)       # Red: Straight
cv2.drawContours(output, [box], 0, (0, 255, 0), 2)              # Green: Rotated
cv2.drawContours(output, [hull], 0, (0, 0, 255), 2)             # Blue: Convex Hull

show_image(output, "Red: Straight | Green: Rotated Rect | Blue: Convex Hull")


## Day 5 — Hough Transforms & Robust Line Fitting
**Topics:** Hough Lines, Hough Circles, Sub-pixel Line Fitting.

### 1. Expert Addition: Sub-Pixel Robust Line Fitting
While `HoughLinesP` is great for finding segments, professionals use statistical M-estimators to fit a single mathematical line through noisy points.
*   **`cv2.fitLine(points, distType, param, reps, aeps)`**: Fits a line using algorithms like L2 (Least Squares) or L1/L12 distance types that reject statistical outliers. Returns a directional vector $(vx, vy)$ and a point $(x, y)$.


In [ ]:
# Generate noisy points along a line
y_vals = np.linspace(50, 250, 50)
x_vals = 0.5 * y_vals + 50 + np.random.normal(0, 5, 50) # Slope 0.5 with noise
points = np.column_stack((x_vals, y_vals)).astype(np.int32)

# Add an extreme outlier
points = np.append(points, [[250, 50]], axis=0)

img_fit = np.zeros((300, 300, 3), dtype=np.uint8)
for p in points: cv2.circle(img_fit, tuple(p), 3, (255, 255, 255), -1)

# Fit robust line (DIST_L1 is highly resistant to outliers)
[vx, vy, x, y] = cv2.fitLine(points, cv2.DIST_L1, 0, 0.01, 0.01)

# Calculate endpoints to draw the infinite line
left_y = int((-x * vy / vx) + y)
right_y = int(((300 - x) * vy / vx) + y)
cv2.line(img_fit, (300, right_y), (0, left_y), (0, 255, 0), 2)

show_image(img_fit, "Robust fitLine (Ignoring Outlier)")


### 2. Hough Circles
*   **`cv2.HoughCircles(..., method=cv2.HOUGH_GRADIENT)`**: Detects circles using the gradient orientation. Much faster than standard parameter space searches.


In [ ]:
img_smarties_gray = cv2.cvtColor(img_smarties, cv2.COLOR_RGB2GRAY)
img_smarties_gray = cv2.medianBlur(img_smarties_gray, 5)

circles = cv2.HoughCircles(img_smarties_gray, cv2.HOUGH_GRADIENT, dp=1, minDist=20,
                           param1=50, param2=30, minRadius=15, maxRadius=40)

out_circles = img_smarties.copy()
if circles is not None:
    circles = np.uint16(np.around(circles))
    for i in circles[0, :]:
        cv2.circle(out_circles, (i[0], i[1]), i[2], (0, 255, 0), 2) # Outer edge
        cv2.circle(out_circles, (i[0], i[1]), 2, (0, 0, 255), 3)    # Center

show_image(out_circles, "Hough Circles Detection")


## Day 6 — Perspective Transform & Automated Homography
**Topics:** Perspective Matrices, Keypoint Extraction, Feature Matching (ORB), RANSAC.

### 1. Expert Addition: Automated Homography via ORB and RANSAC
Manually defining 4 corner points (as done in introductory CV) is useless in automated pipelines. Professionals use local feature descriptors (like ORB, SIFT) to match images.
*   **`cv2.ORB_create()`**: Oriented FAST and Rotated BRIEF. Extracts corners and calculates a binary string descriptor defining the corner's unique visual signature.
*   **`cv2.BFMatcher`**: Brute Force Matcher. Compares signatures between two images using Hamming distance.
*   **`cv2.findHomography(src_pts, dst_pts, cv2.RANSAC)`**: Random Sample Consensus. Mathematically fits the 3x3 transformation matrix while rejecting incorrect keypoint matches (outliers).


In [ ]:
# Simulate a skewed target by applying a manual rotation & perspective transform
pts1 = np.float32([[50,50],[250,50],[50,250],[250,250]])
pts2 = np.float32([[10,100],[200,50],[100,250],[280,210]])
M_fake = cv2.getPerspectiveTransform(pts1, pts2)
img_sudoku_skewed = cv2.warpPerspective(img_sudoku, M_fake, (300, 300))

# --- AUTOMATED HOMOGRAPHY PIPELINE ---
# 1. Initialize ORB
orb = cv2.ORB_create(500)
kp1, des1 = orb.detectAndCompute(img_sudoku, None)
kp2, des2 = orb.detectAndCompute(img_sudoku_skewed, None)

# 2. Match Features
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = bf.match(des1, des2)
matches = sorted(matches, key=lambda x: x.distance)

# 3. Extract matching coordinates
src_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
dst_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

# 4. Calculate robust Homography using RANSAC
M_RANSAC, mask = cv2.findHomography(dst_pts, src_pts, cv2.RANSAC, 5.0)

# 5. Warp back to fix the image automatically!
img_fixed = cv2.warpPerspective(img_sudoku_skewed, M_RANSAC, (img_sudoku.shape[1], img_sudoku.shape[0]))

img_matches = cv2.drawMatches(img_sudoku, kp1, img_sudoku_skewed, kp2, matches[:20], None, flags=2)

show_image(img_matches, "ORB Keypoint Matching", figsize=(12,6))
show_images_grid([img_sudoku_skewed, img_fixed], ["Skewed Image", "Auto-Aligned via ORB + RANSAC"], 1, 2)


## Day 7 — Phase 1 Project (Expert Edition)
**Goal:** Complete Wall / Reference Line Detection Pipeline using Professional Methodologies.

**Expert Upgrades:**
1.  **Noise Mitigation:** We use Bilateral Filtering to maintain hard wall edges while crushing floor noise.
2.  **Edge Detection:** Auto-Canny handles dynamic brightness variations without hardcoded thresholds.
3.  **Line Filtering:** Hough lines extract segments, but we calculate weighted sub-pixel averages to determine the absolute reference line equation.


In [ ]:
def generate_synthetic_wall(angle_deg, distance_from_center, noise_level=10):
    '''Generates ground-truth wall geometry.'''
    img = np.zeros((400, 400), dtype=np.uint8); img[:] = 180
    rad = math.radians(angle_deg)
    nx, ny = math.cos(rad - math.pi/2), math.sin(rad - math.pi/2)
    px, py = 200 + nx * distance_from_center, 200 + ny * distance_from_center
    dx, dy = math.cos(rad), math.sin(rad)
    pt1, pt2 = (int(px + dx*1000), int(py + dy*1000)), (int(px - dx*1000), int(py - dy*1000))
    cv2.fillPoly(img, np.array([[pt1, pt2, (pt2[0], 0), (pt1[0], 0)]], dtype=np.int32), 80)
    return np.clip(img.astype(np.float32) + np.random.normal(0, noise_level, img.shape), 0, 255).astype(np.uint8), pt1, pt2

def expert_wall_pipeline(image):
    '''Expert pipeline combining Bilateral, Auto-Canny, and M-Estimators.'''
    # 1. Edge-preserving noise removal
    filtered = cv2.bilateralFilter(image, 9, 75, 75)
    
    # 2. Dynamic thresholding edge detection
    edges = auto_canny(filtered, sigma=0.4)
    
    # 3. Morphology to seal tiny breaks in the edge
    kernel = np.ones((3,3), np.uint8)
    closed_edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)
    
    # 4. Extract all line segments
    lines = cv2.HoughLinesP(closed_edges, 1, np.pi/180, 50, minLineLength=50, maxLineGap=20)
    
    if lines is None: return None
    
    # 5. Extract coordinate points of near-horizontal lines for robust fitting
    valid_points = []
    for line in lines:
        x1, y1, x2, y2 = line[0]
        if abs(math.degrees(math.atan2(y2-y1, x2-x1))) < 45: # Wall is mostly horizontal
            valid_points.extend([[x1, y1], [x2, y2]])
            
    if not valid_points: return None
    
    # 6. Fit single mathematical line using L1 norm (Highly resistant to outliers)
    valid_points = np.array(valid_points, dtype=np.int32)
    [vx, vy, x, y] = cv2.fitLine(valid_points, cv2.DIST_L1, 0, 0.01, 0.01)
    
    angle = math.degrees(math.atan2(vy[0], vx[0]))
    return vx[0], vy[0], x[0], y[0], angle

# Test Pipeline
img, _, _ = generate_synthetic_wall(17.5, 45, noise_level=20)
result = expert_wall_pipeline(img)

annotated = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
if result:
    vx, vy, x, y, ang = result
    pt1 = (int(x - vx*1000), int(y - vy*1000))
    pt2 = (int(x + vx*1000), int(y + vy*1000))
    cv2.line(annotated, pt1, pt2, (0, 255, 0), 2)
    cv2.putText(annotated, f"Sub-Pixel Angle: {ang:.2f} deg", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

show_image(annotated, "Expert Phase 1 Output via robust fitLine", figsize=(8,8))
